# Versioned external evidence
This independent, fabricated M1 → M2 → M3 example keeps SCV and VCV claims separate. A link is neither carriage nor a clinical conclusion.

In [ ]:
import os, sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
from genome_evidence.notebook_support import resolve_settings
SETTINGS = resolve_settings()
PROFILE = SETTINGS.profile
REPOSITORY_URL = SETTINGS.repository_url
REPOSITORY_REF = SETTINGS.repository_ref
WORKSPACE_ROOT = SETTINGS.workspace_root
SUBJECT_ID = SETTINGS.subject_id
print({"profile": PROFILE, "requested_ref": REPOSITORY_REF})


In [ ]:
if PROFILE == "personal_drive":
    from genome_evidence.workspace import validate_workspace

    validate_workspace(WORKSPACE_ROOT)
else:
    assert PROFILE == "synthetic_ci"

In [ ]:
# ruff: noqa
import json, tempfile
from pathlib import Path
from genome_evidence.ingest import Ingest23andMeConfig, ingest_23andme
from genome_evidence.normalization import NormalizationConfig, normalize_m1_run
from genome_evidence.evidence import ingest_clinvar_vcv, link_external_evidence

root = Path(tempfile.mkdtemp())
source = root / "source.txt"
source.write_text("# genome build: GRCh38\nsynthetic_marker\t1\t5\tAA\n")
markers = root / "markers.json"
markers.write_text(
    json.dumps(
        [
            {
                "marker_id": "synthetic_marker",
                "assembly": "GRCh38",
                "chromosome": "1",
                "position": 5,
                "reference": "A",
                "alternate": "G",
                "orientation": "none",
                "orientation_authoritative": True,
            }
        ]
    )
)
fasta = root / "ref.fa"
fasta.write_text(">1\n" + "A" * 20 + "\n")
ingest_23andme(source, root / "m1", Ingest23andMeConfig(genome_build_override="GRCh38"))
normalize_m1_run(
    root / "m1",
    root / "m2",
    NormalizationConfig(marker_definitions=markers, target_reference=fasta),
)
xml = root / "synthetic.xml"
xml.write_text(
    """<ReleaseSet Dated="2026-07-01" ReleaseID="synthetic-release"><VariationArchive Accession="VCV999999001" Version="1"><ClassifiedRecord><SimpleAllele AlleleID="1"><SequenceLocation Assembly="GRCh38" Chr="1" positionVCF="5" referenceAlleleVCF="A" alternateAlleleVCF="G"/></SimpleAllele><Classifications><GermlineClassification><Description>synthetic aggregate term</Description></GermlineClassification></Classifications><ClinicalAssertion Accession="SCV999999001" Version="1"><Submitter Name="Fabricated Submitter"/><GermlineClassification><Description>synthetic submitted term A</Description></GermlineClassification></ClinicalAssertion><ClinicalAssertion Accession="SCV999999002" Version="1"><GermlineClassification><Description>synthetic submitted term B</Description></GermlineClassification></ClinicalAssertion></ClassifiedRecord></VariationArchive><VariationArchive Accession="VCV999999002" Version="1"><ClassifiedRecord><Haplotype/><Classifications><OncogenicityClassification><Description>synthetic unsupported term</Description></OncogenicityClassification></Classifications></ClassifiedRecord></VariationArchive></ReleaseSet>"""
)
evidence = ingest_clinvar_vcv(xml, root / "evidence")
annotation = link_external_evidence(root / "m2", root / "evidence", root / "annotation")
assert len([a for a in evidence.assertions if a.scv_accession]) == 2
assert len([a for a in evidence.assertions if not a.scv_accession]) == 2
assert {x.outcome.value for x in annotation.links} == {"matched", "unsupported"}
assert all(not hasattr(x, "genotype") for x in annotation.links)
assert "does not establish" in (root / "annotation/annotation_report.md").read_text()
[(a.logical_source_key, a.source_classification_terms) for a in evidence.assertions]